In [1]:

#   - Download official Census boundaries automatically (no CSVs/uploads).
#   - Compute how much of each county belongs to each congressional district.
#   - Compute how much of each congressional district comes from each county.

# 1. Install dependencies (once per runtime)
!pip -q install geopandas pyogrio shapely fiona requests tqdm certifi pyproj

# 2. Imports
import geopandas as gpd
import pandas as pd
import requests, zipfile, io, os, subprocess, shlex, certifi, sys
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter
from pathlib import Path

# 3. Data sources (Census Cartographic Boundary files, nationwide)
DATA_DIR = Path("/content/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

COUNTY_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_500k.zip"
DIST_URL   = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_cd119_500k.zip"
# To use 118th districts instead of 119th, you can uncomment this:
# DIST_URL = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_cd118_500k.zip"

# 4. Download helper with retries + SSL fix + curl fallback
def _requests_session():
    retries = Retry(total=5, backoff_factor=0.8,
                    status_forcelist=[429, 500, 502, 503, 504],
                    allowed_methods=["GET", "HEAD"])
    s = requests.Session()
    s.trust_env = False
    s.headers.update({"User-Agent": "colab-census-fetch/1.0"})
    s.mount("https://", HTTPAdapter(max_retries=retries))
    s.mount("http://", HTTPAdapter(max_retries=retries))
    return s

def download_and_unzip(url: str, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    stamp = out_dir / ".done"
    if not stamp.exists():
        try:
            s = _requests_session()
            resp = s.get(url, stream=True, timeout=180, verify=certifi.where())
            resp.raise_for_status()
            z = zipfile.ZipFile(io.BytesIO(resp.content))
        except Exception as e:
            print(f"[requests] Download failed ({e}). Trying curl fallback...")
            tmp_zip = out_dir / "tmp.zip"
            cmd = f'curl -L --fail --retry 5 --retry-delay 2 -o {shlex.quote(str(tmp_zip))} {shlex.quote(url)}'
            subprocess.check_call(cmd, shell=True)
            z = zipfile.ZipFile(tmp_zip)
        print(f"Unzipping to {out_dir} ...")
        z.extractall(out_dir)
        stamp.write_text("ok")
    for p in out_dir.rglob("*.shp"):
        return p
    raise FileNotFoundError(f"No .shp found after unzipping {url}")

def load_layer(url: str, name: str) -> gpd.GeoDataFrame:
    folder = DATA_DIR / name
    shp = download_and_unzip(url, folder)
    try:
        gdf = gpd.read_file(shp, engine="pyogrio")
    except Exception:
        gdf = gpd.read_file(shp)
    if gdf.crs is None:
        gdf.set_crs(epsg=4269, inplace=True)  # NAD83
    else:
        gdf = gdf.to_crs(epsg=4269)
    return gdf

# 5. Compute proportions (core math logic)
def compute_proportions(counties: gpd.GeoDataFrame, dists: gpd.GeoDataFrame):
    counties["fips"] = counties["STATEFP"].str.zfill(2) + counties["COUNTYFP"].str.zfill(3)
    dists["DistrictID"] = dists["GEOID"].astype(str)

    counties = counties.to_crs(5070)  # equal-area projection
    dists = dists.to_crs(5070)

    counties["county_area_m2"] = counties.geometry.area
    dists["district_area_m2"] = dists.geometry.area

    print("Computing county–district intersections ...")
    inter = gpd.overlay(
        counties[["fips", "geometry"]],
        dists[["DistrictID", "geometry"]],
        how="intersection",
        keep_geom_type=False
    )
    inter["intersect_area_m2"] = inter.geometry.area
    inter = inter.merge(counties[["fips","county_area_m2"]], on="fips", how="left")
    inter = inter.merge(dists[["DistrictID","district_area_m2"]], on="DistrictID", how="left")

    # A) % of county in each district
    county_to_dist = inter.groupby(["fips","DistrictID"], as_index=False).agg(
        intersect_area_m2=("intersect_area_m2","sum"),
        county_area_m2=("county_area_m2","first")
    )
    county_to_dist["pct_of_county_in_district"] = (
        100 * county_to_dist["intersect_area_m2"] / county_to_dist["county_area_m2"]
    )

    # B) % of district from each county
    dist_to_county = inter.groupby(["DistrictID","fips"], as_index=False).agg(
        intersect_area_m2=("intersect_area_m2","sum"),
        district_area_m2=("district_area_m2","first")
    )
    dist_to_county["pct_of_district_from_county"] = (
        100 * dist_to_county["intersect_area_m2"] / dist_to_county["district_area_m2"]
    )

    return county_to_dist, dist_to_county

# 6. Main run
def main():
    print("Loading counties and congressional districts ...")
    counties = load_layer(COUNTY_URL, "counties")
    dists    = load_layer(DIST_URL,   "districts")

    print("Calculating overlaps ...")
    c2d, d2c = compute_proportions(counties, dists)

    out1 = DATA_DIR / "county_to_district_pct.csv"
    out2 = DATA_DIR / "district_from_county_pct.csv"
    c2d.to_csv(out1, index=False)
    d2c.to_csv(out2, index=False)

    print("\n=== Outputs ===")
    print(out1, "\n", out2)
    print("\nSample rows (county_to_district_pct):")
    print(c2d.head().to_string(index=False))
    print("\nSample rows (district_from_county_pct):")
    print(d2c.head().to_string(index=False))

if __name__ == "__main__":
    import ssl
    print("Python:", sys.version.split()[0])
    print("OpenSSL:", ssl.OPENSSL_VERSION)
    print("certifi bundle:", certifi.where())
    main()


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 26.7 MB/s eta 0:00:00
Python: 3.12.11
OpenSSL: OpenSSL 3.0.2 15 Mar 2022
certifi bundle: /usr/local/lib/python3.12/dist-packages/certifi/cacert.pem
Loading counties and congressional districts ...


[requests] Download failed (HTTPSConnectionPool(host='www2.census.gov', port=443): Max retries exceeded with url: /geo/tiger/GENZ2024/shp/cb_2024_us_county_500k.zip (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))). Trying curl fallback...
Unzipping to /content/data/counties ...


[requests] Download failed (HTTPSConnectionPool(host='www2.census.gov', port=443): Max retries exceeded with url: /geo/tiger/GENZ2024/shp/cb_2024_us_cd119_500k.zip (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)')))). Trying curl fallback...
Unzipping to /content/data/districts ...
Calculating overlaps ...
Computing county–district intersections ...

=== Outputs ===
/content/data/county_to_district_pct.csv 
 /content/data/district_from_county_pct.csv

Sample rows (county_to_district_pct):
 fips DistrictID  intersect_area_m2  county_area_m2  pct_of_county_in_district
01001       0102       0.000000e+00    1.565324e+09                        0.0
01001       0106       1.565324e+09    1.565324e+09                      100.0
01001       0107       0.000000e+00    1.565324e+09                        0.0
01003       0101       4.350730e+09    4.350730e+09                      100

In [4]:

!git clone https://github.com/policy-design-lab/pdl-api.git
%cd pdl-api

!git checkout -b feature/county-district-overlap


Cloning into 'pdl-api'...
remote: Enumerating objects: 1375, done.
remote: Counting objects: 100% (725/725), done.
remote: Compressing objects: 100% (337/337), done.
remote: Total 1375 (delta 615), reused 388 (delta 388), pack-reused 650 (from 2)
Receiving objects: 100% (1375/1375), 60.93 MiB | 22.51 MiB/s, done.
Resolving deltas: 100% (872/872), done.
/content/pdl-api/pdl-api
Switched to a new branch 'feature/county-district-overlap'


In [5]:
!cp /content/districtMath.ipynb /content/pdl-api/



cp: cannot stat '/content/districtMath.ipynb': No such file or directory
